# How far may a warm start sit from the solution?

[C01](../C_gauge_symmetries_and_NDAR/C01_run_qaoa_optimization_with_gauges.ipynb) swept random
bitflip gauges and related the quality of a NOISY optimization to the energy that |00...0> has
under each one. This notebook asks the same question of a WARM-STARTED optimization, and needs
no noise at all to answer it: a warm-started ansatz is biased towards |00...0>, so the gauge
decides which bitstring the run is biased towards.

The sweep here is over the Hamming distance `d` between the warm start and the ground state,
and it is repeated for several strengths of the warm-start bias, one curve each.
At `d = 0` the warm start IS the solution; at `d = n / 2` it is as far from it as a random
bitstring.

That curve is the main rationale behind NDAWS:
[D02](D02_run_qaoa_optimization_with_NDAWS_qiskit.ipynb) puts a warm-started QAOA inside the
NDAR loop, which re-gauges after every iteration so that the warm start keeps moving towards
the best solution found so far -- that is, towards `d = 0` on this curve.

**What to change**: `number_of_qubits` and `seed_cost_hamiltonian` in the instance cell;
`hamming_distances` (which distances are sampled); `number_of_function_calls` and
`ws_bias_parameters` in the optimization cell.

**Writes**: the Hamiltonian instance and its known solutions into the standardized result store
(`DEFAULT_STORAGE_DIRECTORY`; with that variable unset it is `output/` at the repository
root), and the figure as an HTML file into `../temp/`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from quapopt.ancillary_functions.presets import generate_random_hamiltonian
from quapopt.data_analysis.data_handling import HamiltonianModels
from quapopt.optimization.QAOA.simulation.QAOARunnerExpValuesRDMs import QAOARunnerExpValuesRDMs
from quapopt.optimization.parameter_setting.variational.QAOAOptimizationRunner import QAOAOptimizationRunner
from quapopt.optimization.parameter_setting.variational.scipy_tools.ScipyOptimizerWrapped import ScipyOptimizerWrapped

os.makedirs('../temp', exist_ok=True)

### The warm-started ansatz, in one cell

* A warm-started ansatz biases the initial product state towards a candidate solution instead
  of starting from the uniform superposition. `ws_bias_parameters` is the per-qubit weight $c$
  of that bias, the probability of reading $1$ on that qubit: $c = 0.5$ is no bias at all, and
  smaller values sit closer to the warm start $|0\dots 0\rangle$.
* With $\theta = 2\arcsin\sqrt{c}$, the ansatz this repository implements is

$$|c\rangle = R_Y(\theta)|0\rangle = \sqrt{1-c}\,|0\rangle + \sqrt{c}\,|1\rangle ,$$

$$H_M = \sin\theta\, X + \cos\theta\, Z = 2\sqrt{c(1-c)}\, X + (1-2c)\, Z ,$$

$$U_M(\beta) = e^{-i\beta H_M} = R_Y(\theta)\, R_Z(2\beta)\, R_Y(-\theta) .$$

  The initial state and the mixer axis are tilted TOGETHER: $|c\rangle$ is an eigenstate of
  $H_M$.

At $c = 0.5$ this is $|+\rangle$ with $H_M = X$, which is vanilla QAOA.
* The warm start is ALWAYS $|0\dots 0\rangle$. Moving the instance to a different bitflip
  gauge is therefore how you choose which bitstring the ansatz is biased towards, and that is
  the one knob this notebook turns.
* [B03](../B_running_qaoa_circuits/B03_run_single_layer_qaoa_expected_values.ipynb) runs this
  same ansatz on a larger instance and compares two ways of spending an angle-search budget.


### The instance, and one gauge per distance

* For each Hamming distance `d` the cell below builds the gauge that maps the ground state to
  a bitstring `d` flips away from |00...0>, so that a warm-started run on it starts `d` flips
  from the solution.
* The instance carries no local fields, so its spectrum is symmetric under flipping every
  bit: the complement of the ground state is a ground state of the same energy. The warm
  start is therefore `d` flips from one ground state and `n - d` flips from the other, and
  what the optimization can feel is the SMALLER of the two. The table below reports that
  effective distance beside the nominal one, and the sweep goes all the way to `d = n` so
  that the symmetry is visible rather than asserted.
* **What to change here**: `number_of_qubits`, `seed_cost_hamiltonian` and
  `hamming_distances`. Keep `number_of_qubits` at 25 or fewer: up to 25 qubits
  `solve_hamiltonian` enumerates every state, so the state the gauges start from is the exact
  ground state. Above 25 it is the best state a heuristic found.
* **Result**: `gauges_by_distance`, one gauge per distance. The cell prints the instance, its
  extremal energies and the distances.


In [ ]:
#One instance for the whole sweep. At 25 qubits or fewer the solve below is exact; above that
#it falls back to the Burer-Monteiro heuristic and returns the best state it found.
number_of_qubits = 24
seed_cost_hamiltonian = 0

cost_hamiltonian = generate_random_hamiltonian(
    hamiltonian_model=HamiltonianModels.SherringtonKirkpatrick,
    number_of_qubits=number_of_qubits,
    seed=seed_cost_hamiltonian)

#The gauges are defined by flipping bits of the ground state, so the extremal energies and the
#ground state itself have to be known first. They are cached in the store after the first run.
if cost_hamiltonian.lowest_energy is None:
    cost_hamiltonian.solve_hamiltonian(both_directions=True)

lowest_energy = float(cost_hamiltonian.lowest_energy)
highest_energy = float(cost_hamiltonian.highest_energy)
ground_state = np.array(cost_hamiltonian.ground_state, dtype=int)

print("Instance:", cost_hamiltonian.hamiltonian_instance_description)
print("Ground state energy:", lowest_energy, " highest energy:", highest_energy)

#The gauge that maps the ground state to a bitstring at Hamming distance d from |00...0> is
#the ground state with d of its bits flipped.
#d = 0 puts the warm start exactly on the ground state; d = n flips every bit of it.
hamming_distances = [0, 1, 3, 6, 12, 18, number_of_qubits]
rng = np.random.default_rng(seed=seed_cost_hamiltonian)

gauges_by_distance = {}
for distance in hamming_distances:
    indices_to_flip = rng.choice(number_of_qubits, size=distance, replace=False)
    gauge_bitstring = ground_state.copy()
    gauge_bitstring[indices_to_flip] = 1 - gauge_bitstring[indices_to_flip]
    gauges_by_distance[distance] = tuple(int(bit) for bit in gauge_bitstring)

#A sanity check that the construction does what the prose says: under its own gauge, the
#ground state sits exactly `distance` flips away from the all-zeros state.
for distance, gauge in gauges_by_distance.items():
    assert int(np.sum(ground_state ^ np.array(gauge, dtype=int))) == distance
print("gauges built, at Hamming distances:", list(gauges_by_distance))

### Run the warm-started optimization once per gauge and bias

* Each gauge gets its own gauged copy of the instance, and each copy runs the same single-layer
  warm-started optimization once per warm-start bias $c$. The expectation values come from the
  reduced-density-matrix simulator, so no sampling and no noise enter anywhere.
* The mixer angle is found numerically for each phase angle (`find_best_beta_only=True`), so
  one call means one phase angle and the search is one-dimensional.
* The approximation ratio uses the extremal energies of the ORIGINAL instance, because a
  bitflip gauge does not move the spectrum.
* **What to change here**: `number_of_function_calls` (the budget per run),
  `ws_bias_parameters` (which warm-start biases are swept) and the optimizer settings. The
  cell costs one optimization per bias per distance, so both lists set the wall time.
* **Result**: `distance_study_dataframe`, one row per (bias, distance) pair, with the columns
  `WarmStartBias`, `HammingDistance`, `EffectiveDistance`, `BestEnergy` and
  `ApproximationRatio`. The cell shows it as ratios laid out distance by bias.

In [ ]:
number_of_function_calls = 60
#The warm-start bias c of the section above, one curve per value. c = 0.5 is no bias at all,
#i.e. vanilla QAOA, and is the control: without a bias the gauge cannot matter.
ws_bias_parameters = [0.5, 0.35, 0.2, 0.1]

#One parameter again: the mixer angle is found numerically for each phase angle.
#The seed keeps basinhopping's random displacement steps the same from run to run.
classical_optimizer = ScipyOptimizerWrapped(
    parameters_bounds=[(0, np.pi)],
    optimizer_name='COBYQA',
    optimizer_kwargs=None,
    basinhopping=True,
    basinhopping_kwargs={'niter': 2, 'seed': 0},
    starting_point=[0.0001])

#The gauged copies are built once and reused by every bias.
hamiltonians_by_distance = {distance: cost_hamiltonian.copy().apply_bitflip(bitflip_tuple=gauge)
                            for distance, gauge in gauges_by_distance.items()}

distance_study_results = []
sweep = [(bias, distance) for bias in ws_bias_parameters for distance in hamiltonians_by_distance]
for ws_bias_parameter, distance in tqdm(sweep, desc='bias x gauge', colour='blue'):
    qaoa_runner = QAOARunnerExpValuesRDMs(
        hamiltonian_representations_cost=[hamiltonians_by_distance[distance]],
        store_full_information_in_history=False,
        ws_bias_parameters=ws_bias_parameter)
    qaoa_optimizer = QAOAOptimizationRunner(qaoa_runner)

    best_result, _ = qaoa_optimizer.run_optimization(
        qaoa_depth=1,
        number_of_function_calls=number_of_function_calls,
        classical_optimizer=classical_optimizer,
        measurement_noise=None,
        show_progress_bar=False,
        verbosity=0,
        find_best_beta_only=True)

    best_energy = float(best_result[0][0])
    #A bitflip gauge does not move the spectrum, so the original instance's extremal energies
    #are the right reference for every gauge.
    approximation_ratio = ((highest_energy - best_energy)
                           / (highest_energy - lowest_energy))
    #Without local fields the ground state is doubly degenerate, so the warm start is as
    #close to the complement as `n - d` flips. The effective distance is the smaller one.
    if 1 in cost_hamiltonian.localities:
        effective_distance = distance
    else:
        effective_distance = min(distance, number_of_qubits - distance)

    distance_study_results.append({'WarmStartBias': ws_bias_parameter,
                                   'HammingDistance': distance,
                                   'EffectiveDistance': effective_distance,
                                   'BestEnergy': best_energy,
                                   'ApproximationRatio': approximation_ratio})

distance_study_dataframe = pd.DataFrame(distance_study_results)
distance_study_dataframe.pivot(index='HammingDistance', columns='WarmStartBias',
                               values='ApproximationRatio')

### The curves

* One curve per warm-start bias $c$: the approximation ratio against the Hamming distance `d`
  between the warm start and the ground state, swept from `0` to `n`.
* Each curve falls until `d = n / 2` and then climbs back to its starting value at `d = n`.
  That shape is the Z2 symmetry of a field-free instance: at `d = n` the warm start sits on
  the COMPLEMENT of the ground state, which has the same energy, so the optimization is in
  exactly the situation it was in at `d = 0`. What the ansatz feels is the effective distance
  `min(d, n - d)`, printed as its own column in the table above. Two gauges at the same
  effective distance flip different bits, so their ratios agree only approximately; the
  equality is exact for the pair `d = 0`, `d = n`.
* The $c = 0.5$ curve is the control, and it is flat: with no bias the ansatz is vanilla QAOA,
  which sees the same landscape under every bitflip gauge. The stronger the bias, the more the
  curve moves -- up where the warm start is close to a solution, down where it is far from
  both.
* A warm start that is merely CLOSE to the solution is already worth most of the gain, and
  that is what makes the NDAR loop of
  [D02](D02_run_qaoa_optimization_with_NDAWS_qiskit.ipynb) pay: it never needs the solution,
  only a series of ever-closer guesses to re-gauge towards.
* **What to change here**: nothing is needed; the traces and the axes follow the study above.
* **Result**: the figure, shown here and written to `../temp/D01_warm_start_distance.html`.

In [ ]:
import plotly
import plotly.graph_objects as go

plotly.io.templates.default = "plotly"
plotly.offline.init_notebook_mode(connected=True)

#One trace per bias, against the nominal distance. The symmetry about n/2 is the point of the
#figure, so the sweep is not folded onto min(d, n - d): that column stays in the table instead.
distance_figure = go.Figure()
for ws_bias_parameter, rows in distance_study_dataframe.groupby('WarmStartBias', sort=False):
    distance_figure.add_trace(go.Scatter(x=rows['HammingDistance'].tolist(),
                                         y=rows['ApproximationRatio'].tolist(),
                                         mode='lines+markers',
                                         customdata=rows['EffectiveDistance'].tolist(),
                                         hovertemplate=('d = %{x}, effective %{customdata}'
                                                        '<br>approximation ratio %{y:.4f}<extra></extra>'),
                                         name=f'c = {ws_bias_parameter}'))
distance_figure.update_layout(
    title=(f"Warm-started p=1 QAOA against the distance to the solution, "
           f"{number_of_qubits} qubits, "
           f"{cost_hamiltonian.hamiltonian_class_description}"),
    xaxis_title="Hamming distance between the warm start and the ground state",
    yaxis_title="Approximation Ratio",
    legend_title="warm-start bias",
    template="plotly",
    font=dict(size=14),
    hovermode="x unified")
distance_figure.update_xaxes(tickvals=hamming_distances)

plotly.offline.plot(distance_figure, filename='../temp/D01_warm_start_distance.html')
# distance_figure.show()